
# Fixed Scaled-Release Policy Ablation — Fully Fixed Version

This notebook evaluates simple fixed ramp-release policies using the **fixed benchmark source of truth** from `shared_benchmark_inputs.pkl`.

The fixed policy is intentionally simple:

$[
u_r(t) = s\,u_r^{obs}(t)
]$

where `s` is a uniform release scale and $(u_r^{obs}(t))$ is the observed 15-second ramp release profile from the benchmark export.


In [12]:

from pathlib import Path
import pickle

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

EXPECTED_BENCHMARK_VERSION = "fixed_2026_06_17"
EXPECTED_ARRIVAL_MULTIPLIER = 1.2

# Pull the fixed benchmark by running the benchmark notebook in-place.
# shared_benchmark_inputs (and also refreshes shared_benchmark_inputs.pkl).
%run Benchmark_calculation.ipynb

if "shared_benchmark_inputs" not in globals():
    raise RuntimeError(
        "%run Benchmark_calculation.ipynb did not define shared_benchmark_inputs. "
        "Make sure Benchmark_calculation.ipynb is in the same folder as this notebook."
    )

shared = shared_benchmark_inputs
shared_inputs_path = "Benchmark_calculation.ipynb (via %run)"


def load_pickle(path):
    with open(path, "rb") as file:
        return pickle.load(file)


print("Loaded fixed benchmark via %run Benchmark_calculation.ipynb")


def get_input(name):
    if name not in shared:
        raise KeyError(f"{name} missing from {shared_inputs_path}")
    return shared[name]


def get_optional_input(name, default_value=None):
    return shared[name] if name in shared else default_value

cell_ids = ["Cell %d" % i for i in range(1, 10)]
ramp_ids = get_input("ramp_ids")

num_steps = int(get_input("num_steps"))
delta_t = float(get_input("delta_t"))

q_in_boundary_series = get_input("q_in_boundary_series")
observed_release_series = get_input("observed_release_series")
f_out_series = get_input("f_out_series")
undetected_entry_per_15sec_by_cell = get_input("undetected_entry_per_15sec_by_cell")

mainline_initial_state = get_input("mainline_initial_state")
ramp_queue_0 = get_input("ramp_queue_0")
external_queue_0 = get_input("external_queue_0")

inflow_capacity = get_input("inflow_capacity")
outflow_capacity = get_input("outflow_capacity")
physical_capacity = get_input("physical_capacity")
safe_threshold_capacity = get_input("safe_threshold_capacity")

movement_factor_by_cell = get_input("movement_factor_by_cell")
wave_speed_ratio_by_cell = get_input("wave_speed_ratio_by_cell")
exit_split_by_cell = get_input("exit_split_by_cell")

generic_ramp_cell_map = get_input("generic_ramp_cell_map")
ramp_cell_map = get_input("ramp_cell_map")
merge_ramp_id = get_input("merge_ramp_id")
merge_ramp_cell = get_input("merge_ramp_cell")
MERGE_PRIORITY = float(get_input("MERGE_PRIORITY"))

ramp_name_map = get_input("ramp_name_map")
ramp_max_queue_named = get_input("ramp_max_queue_named")
ramp_max_queue_by_u = get_input("ramp_max_queue_by_u")

tt_ff_min = get_input("tt_ff_min")
use_clipped_mainline_delay = bool(get_optional_input("use_clipped_mainline_delay", True))

gamma = float(get_input("gamma"))
lambda_1 = float(get_input("lambda_1"))
lambda_2 = float(get_input("lambda_2"))
lambda_3 = float(get_input("lambda_3"))
lambda_4 = float(get_input("lambda_4"))

official_totals = get_input("official_totals")
official_service_metrics = get_input("official_service_metrics")
shared_arrival_multiplier = float(get_input("arrival_multiplier"))

assert cell_ids == [f"Cell {i}" for i in range(1, 10)]
assert len(ramp_ids) == 4
assert num_steps == 480
assert abs(delta_t - 0.25) < 1e-12
assert merge_ramp_id == "u_avila"
assert merge_ramp_cell == "Cell 9"
assert abs(shared_arrival_multiplier - 1.2) < 1e-12

print("Fixed benchmark verified.")
print("Benchmark version:", shared.get("version"))
print("Benchmark arrival multiplier:", shared_arrival_multiplier)
print("Official benchmark mainline delay:", official_totals["mainline_delay"])
print("Official benchmark raw objective:", official_totals["raw_objective"])


Loading metadata file: C:\Users\User\Desktop\HighwayProject\HighwayProject\d05_text_meta_2026_04_28.txt
Official model IDs loaded.
Cells: ['Cell 1', 'Cell 2', 'Cell 3', 'Cell 4', 'Cell 5', 'Cell 6', 'Cell 7', 'Cell 8', 'Cell 9']
Segments: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Ramps: ['u_4th', 'u_price', 'u_mattie', 'u_avila']
Ramp names: {'u_4th': '4TH ST ON', 'u_price': 'PRICE ST ON', 'u_mattie': 'MATTIE RD ON', 'u_avila': 'AVILA BEACH ON'}
Official ramp mapping
generic_ramp_cell_map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6'}
merge_ramp_id: u_avila
merge_ramp_cell: Cell 9
ramp_cell_map: {'u_4th': 'Cell 2', 'u_price': 'Cell 2', 'u_mattie': 'Cell 6', 'u_avila': 'Cell 9'}
Selected Station Metadata
Number of selected stations: 15


,station_id,station_name,station_type,freeway,direction,absolute_postmile,length,lanes,latitude,longitude
0,501015153,4TH ST 101 NB EXIT VDS MLSB SB,ML,101,S,188.738,0.506,2,35.133413,-120.616280
1,501016013,4TH ST 101 NB ON RAMP VDS MLSB S,ML,101,S,189.253,0.481,2,35.136794,-120.624363
2,501016014,4TH ST 101 NB ON RAMP VDS ONSB S,OR,101,S,189.254,NaN,1,35.136798,-120.624380
3,501016023,PRICE ST EXIT SIGN 101 NB VDS ML,ML,101,S,189.702,0.475,2,35.138331,-120.632049
4,501016024,PRICE ST EXIT SIGN 101 NB VDS ON,OR,101,S,189.703,NaN,1,35.138334,-120.632066
5,501016031,HINDS AVE 101 SB VDS MLSB SB,ML,101,S,190.204,0.506,2,35.141950,-120.639431
6,501016043,BELLO ST 101 NB VDS MLSB SB,ML,101,S,190.714,0.623,2,35.147275,-120.645623
7,501016044,BELLO ST 101 NB VDS OFSB SB,FR,101,S,190.715,NaN,1,35.147282,-120.645638
8,501016053,SHELL BEACH RD 101 NB VDS MLSB S,ML,101,S,191.451,0.540,2,35.151759,-120.657382
9,501016062,MATTIE RD 101 NB VDS MLSB SB,ML,101,S,191.796,0.935,2,35.154187,-120.662680


Official mainline segments


,segment,cell,from_id,to_id,from_station,to_station,from_postmile,to_postmile,length_miles
0,S1,Cell 1,501015153,501016013,4TH ST 101 NB EXIT VDS MLSB SB,4TH ST 101 NB ON RAMP VDS MLSB S,188.738,189.253,0.515
1,S2,Cell 2,501016013,501016023,4TH ST 101 NB ON RAMP VDS MLSB S,PRICE ST EXIT SIGN 101 NB VDS ML,189.253,189.702,0.449
2,S3,Cell 3,501016023,501016031,PRICE ST EXIT SIGN 101 NB VDS ML,HINDS AVE 101 SB VDS MLSB SB,189.702,190.204,0.502
3,S4,Cell 4,501016031,501016043,HINDS AVE 101 SB VDS MLSB SB,BELLO ST 101 NB VDS MLSB SB,190.204,190.714,0.510
4,S5,Cell 5,501016043,501016053,BELLO ST 101 NB VDS MLSB SB,SHELL BEACH RD 101 NB VDS MLSB S,190.714,191.451,0.737
5,S6,Cell 6,501016053,501016062,SHELL BEACH RD 101 NB VDS MLSB S,MATTIE RD 101 NB VDS MLSB SB,191.451,191.796,0.345
6,S7,Cell 7,501016062,501016071,MATTIE RD 101 NB VDS MLSB SB,SPYGLASS DR 101 SB VDS MLSB SB,191.796,193.322,1.526
7,S8,Cell 8,501016071,501016082,SPYGLASS DR 101 SB VDS MLSB SB,AVILA BEACH DR 101 NB VDS MLSB S,193.322,194.463,1.141
8,S9,Cell 9,501016082,501016091,AVILA BEACH DR 101 NB VDS MLSB S,SAN LUIS BAY DR 101 SB VDS MLSB,194.463,195.520,1.057


station_pm:
{501015153: 188.738, 501016013: 189.253, 501016014: 189.254, 501016023: 189.702, 501016024: 189.703, 501016031: 190.204, 501016043: 190.714, 501016044: 190.715, 501016053: 191.451, 501016062: 191.796, 501016063: 191.797, 501016071: 193.322, 501016082: 194.463, 501016083: 194.464, 501016091: 195.52}
station_lane_count:
{501015153: 2, 501016013: 2, 501016014: 1, 501016023: 2, 501016024: 1, 501016031: 2, 501016043: 2, 501016044: 1, 501016053: 2, 501016062: 2, 501016063: 1, 501016071: 2, 501016082: 3, 501016083: 1, 501016091: 2}
PASS: Cell 1 metadata / official geometry loaded cleanly.
Loading: C:\Users\User\Desktop\HighwayProject\HighwayProject\d05_text_station_5min_2026_05_27.txt
Selected single-day corridor data
Benchmark source file: d05_text_station_5min_2026_05_27.txt
Benchmark date: 2026-05-27
Rows: 4320
Unique dates: 1
Unique timestamps: 288
Unique stations: 15


...

Saved shared_benchmark_inputs.pkl
Max official total export mismatch: 0.0
Max denominator export mismatch: 0.0
Max service metric export mismatch: 0.0
Max validation metric export mismatch: 0.0
PASS: shared_benchmark_inputs.pkl matches current official benchmark.
Corrected compact benchmark complete.
arrival_multiplier = 1.2
CAP9_VPH = 2400.0
raw_objective = 130422.809368
max_normal_ramp_control_state_delta_when_zeroed = 283.787050324536


,metric,value
0,mainline_delay,25744.199317
1,mainline_delay_raw,25543.704003
2,mainline_delay_clipped,25744.199317
3,upstream_boundary_delay,0.000000
4,local_delay,44806.729702
5,fairness_penalty,143.623653
6,doorway_penalty,0.000000
7,safe_penalty,0.000000
8,physical_penalty,0.000000
9,spillback_penalty,59728.256696


,metric,value
0,initial_physical_ramp_queue_R,0.000000
1,initial_external_spillback_queue_B,0.000000
2,total_ramp_arrivals,4387.200000
3,total_requested_release,3656.000000
4,total_actual_release,3649.200159
5,final_physical_ramp_queue_R,146.257223
6,final_external_spillback_queue_B,591.742619
7,final_upstream_boundary_queue,0.000000
8,ramp_mass_residual,-0.000000
9,total_ramp_demand_to_account,4387.200000


,metric,value
0,mainline_mass_residual,-0.000000e+00
1,upstream_boundary_mass_residual,0.000000e+00
2,ramp_mass_residual,-1.000000e-12
3,max_receiving_violation,0.000000e+00
4,max_avila_merge_violation,0.000000e+00
5,max_controlled_ramp_leakage_into_external_inflow,0.000000e+00
6,max_normal_ramp_control_state_delta_when_zeroed,2.837871e+02


Loaded fixed benchmark via %run Benchmark_calculation.ipynb
Fixed benchmark verified.
Benchmark version: None
Benchmark arrival multiplier: 1.2
Official benchmark mainline delay: 25744.19931651336
Official benchmark raw objective: 130422.80936777758



## Helper functions

These functions intentionally mirror the fixed benchmark definitions. The two most important objective fixes are:

\[
\phi_r(t) = \frac{R_r(t)+B_r(t)}{R_r^{max}}
\]

for fairness stress, and

\[
\max(0, d_i^{doorway}(t)-C_i)^2
\]

for doorway-capacity pressure, where \(d_i^{doorway}\) is attempted doorway demand **before** CTM acceptance.


In [13]:
# Fixed-policy simulator.

def build_ramp_arrival_series(arrival_multiplier):
    return {
        ramp: [arrival_multiplier * float(value) for value in observed_release_series[ramp]]
        for ramp in ramp_ids
    }


def build_fixed_command_series(release_scale):
    return {
        ramp: [release_scale * float(value) for value in observed_release_series[ramp]]
        for ramp in ramp_ids
    }


def simulate_fixed_policy(release_scale, arrival_multiplier):
    """Run the EXACT benchmark simulator with a scaled, fixed ramp-command schedule.

    """
    commanded_release_series_case = build_fixed_command_series(release_scale)
    ramp_arrival_series_case = build_ramp_arrival_series(arrival_multiplier)

    # Use the benchmark's exact official-run inputs (the *_official capacities/factors and the
    # benchmark's own exogenous series), so reproduction is bit-for-bit at scale 1.0.
    return simulate_state_based_benchmark_480_steps(
        num_steps=num_steps,
        mainline_initial_state=mainline_initial_state,
        ramp_queue_0=ramp_queue_0,
        q_in_boundary_series=q_in_boundary_series,
        commanded_release_series=commanded_release_series_case,
        ramp_arrival_series=ramp_arrival_series_case,
        external_inflow_series=external_inflow_series,
        fixed_outflow_series=fixed_outflow_series,
        inflow_capacity=ctm_inflow_capacity_official,
        outflow_capacity=ctm_outflow_capacity_official,
        physical_capacity=physical_capacity,
        safe_threshold_capacity=safe_threshold_capacity,
        ramp_name_map=ramp_name_map,
        ramp_max_queue_named=ramp_max_queue_named,
        ramp_max_queue_by_u=ramp_max_queue_by_u,
        tt_ff_min=tt_ff_min,
        delta_t=delta_t,
        gamma=gamma,
        lambda_1=lambda_1,
        lambda_2=lambda_2,
        lambda_3=lambda_3,
        lambda_4=lambda_4,
        movement_factor_by_cell=movement_factor_by_cell_official,
        wave_speed_ratio_by_cell=wave_speed_ratio_by_cell_official,
        exit_split_by_cell=exit_split_by_cell,
        external_queue_0=external_queue_0_export,
        use_clipped_mainline_delay=use_clipped_mainline_delay,
    )


In [14]:
# Aggregation helpers

def compute_totals_from_history(history):
    totals = {
        "mainline_delay": float(sum(history["mainline_delay"])),
        "mainline_delay_raw": float(sum(history["mainline_delay_raw"])),
        "mainline_delay_clipped": float(sum(history["mainline_delay_clipped"])),
        "upstream_boundary_delay": float(sum(history["upstream_boundary_delay"])),
        "local_delay": float(sum(history["local_delay"])),
        "fairness_penalty": float(sum(history["fairness_penalty"])),
        "doorway_penalty": float(sum(history["doorway_penalty"])),
        "safe_penalty": float(sum(history["safe_penalty"])),
        "physical_penalty": float(sum(history["physical_penalty"])),
        "spillback_penalty": float(sum(history["spillback_penalty"])),
    }
    totals["capacity_penalty"] = (
        totals["doorway_penalty"]
        + totals["safe_penalty"]
        + totals["physical_penalty"]
        + totals["spillback_penalty"]
    )
    totals["raw_objective"] = (
        totals["mainline_delay"]
        + totals["local_delay"]
        + totals["fairness_penalty"]
        + totals["capacity_penalty"]
    )
    totals["sum_history_total_objective"] = float(sum(history["total_objective"]))
    return totals


def compute_service_metrics(history, arrival_multiplier):
    ramp_arrival_series_case = build_ramp_arrival_series(arrival_multiplier)

    total_initial_R = sum(float(ramp_queue_0.get(ramp, 0.0)) for ramp in ramp_ids)
    total_initial_B = sum(float(external_queue_0.get(ramp, 0.0)) for ramp in ramp_ids)
    total_arrivals = sum(
        float(ramp_arrival_series_case[ramp][step])
        for ramp in ramp_ids
        for step in range(num_steps)
    )
    total_actual_release = sum(
        float(history["actual_release"][step][ramp])
        for ramp in ramp_ids
        for step in range(num_steps)
    )
    final_R = sum(float(history["R_final"][ramp]) for ramp in ramp_ids)
    final_B = sum(float(history["B_final"][ramp]) for ramp in ramp_ids)
    demand_to_account = total_initial_R + total_initial_B + total_arrivals
    ramp_mass_residual = demand_to_account - total_actual_release - final_R - final_B
    served_fraction = total_actual_release / demand_to_account if demand_to_account > 0 else 1.0

    return {
        "initial_physical_ramp_queue_R": float(total_initial_R),
        "initial_external_spillback_queue_B": float(total_initial_B),
        "total_ramp_arrivals": float(total_arrivals),
        "total_actual_release": float(total_actual_release),
        "final_physical_ramp_queue_R": float(final_R),
        "final_external_spillback_queue_B": float(final_B),
        "ramp_mass_residual": float(ramp_mass_residual),
        "total_ramp_demand_to_account": float(demand_to_account),
        "served_fraction": float(served_fraction),
        "final_upstream_boundary_queue": float(history["upstream_boundary_queue_final"]),
    }


def run_policy_grid(arrival_multiplier, release_scales, experiment_name):
    histories = {}
    rows = []

    for release_scale in release_scales:
        release_scale = float(release_scale)
        history = simulate_fixed_policy(
            release_scale=release_scale,
            arrival_multiplier=float(arrival_multiplier),
        )
        totals = compute_totals_from_history(history)
        service = compute_service_metrics(history, arrival_multiplier=float(arrival_multiplier))
        histories[release_scale] = history
        rows.append({
            "experiment": experiment_name,
            "arrival_multiplier": float(arrival_multiplier),
            "release_scale": release_scale,
            **totals,
            **service,
        })

    df = pd.DataFrame(rows)
    baseline_rows = df[df["release_scale"].round(10) == 1.0]
    if baseline_rows.empty:
        raise ValueError("release_scales must include 1.0 so benchmark-change columns can be computed.")
    baseline_row = baseline_rows.iloc[0]

    for metric in [
        "mainline_delay",
        "local_delay",
        "fairness_penalty",
        "doorway_penalty",
        "spillback_penalty",
        "capacity_penalty",
        "raw_objective",
        "served_fraction",
    ]:
        base_value = float(baseline_row[metric])
        df[f"{metric}_change"] = df[metric] - base_value
        if abs(base_value) > 1e-12:
            df[f"{metric}_change_pct"] = 100.0 * df[f"{metric}_change"] / base_value
        else:
            df[f"{metric}_change_pct"] = np.nan

    if df["ramp_mass_residual"].abs().max() >= 1e-8:
        raise AssertionError(f"Ramp mass conservation failed in {experiment_name}.")

    return df, histories



## Reproduction check

Before running any fixed-policy ablation, this notebook must reproduce the fixed benchmark exactly at:

```text
arrival_multiplier = 1.2
release_scale = 1.0
```

That is the benchmark replay policy: estimated arrivals are 20% above observed release, but the no-control release command is the observed ramp release.


In [15]:
# Reproduction check: at release_scale = 1.0 and the official arrival multiplier (1.2),

reproduction_history = simulate_fixed_policy(
    release_scale=1.0,
    arrival_multiplier=shared_arrival_multiplier,
)
reproduction_totals = compute_totals_from_history(reproduction_history)

# Compare two ways:
#  (a) vs the benchmark's own official run, recomputed with the SAME aggregator  -> must be ~0
#  (b) vs the published official_totals dict (sanity; only keys it actually contains)
official_run_totals = compute_totals_from_history(official_benchmark_history)

rows = []
for key in reproduction_totals:
    run_val = official_run_totals.get(key, float("nan"))
    pub_val = official_totals.get(key, float("nan"))
    rows.append({
        "metric": key,
        "fixed_policy_scale1": reproduction_totals[key],
        "benchmark_run": run_val,
        "published_official": pub_val,
        "diff_vs_benchmark_run": reproduction_totals[key] - run_val,
    })
reproduction_check_df = pd.DataFrame(rows)
display(reproduction_check_df.round(8))

max_abs_reproduction_diff = float(reproduction_check_df["diff_vs_benchmark_run"].abs().max())
print("Max abs reproduction diff vs benchmark run:", max_abs_reproduction_diff)
if max_abs_reproduction_diff >= 1e-8:
    display(
        reproduction_check_df.assign(abs=lambda d: d["diff_vs_benchmark_run"].abs())
        .sort_values("abs", ascending=False).head(12).round(8)
    )
    raise AssertionError(
        "Fixed-policy wrapper does not reproduce the benchmark engine. "
        "Check that %run loaded Benchmark_calculation.ipynb and the *_official inputs."
    )

# sanity vs the published raw objective (key is guaranteed present)
print("fixed-policy raw objective (scale 1.0) = %.6f" % reproduction_totals["raw_objective"])
print("published benchmark  raw objective     = %.6f" % official_totals["raw_objective"])
print("PASS: fixed policy reproduces the benchmark exactly (receiving-aware CTM, LINEAR spillback).")


,metric,fixed_policy_scale1,benchmark_run,published_official,diff_vs_benchmark_run
0,mainline_delay,25744.199317,25744.199317,25744.199317,0.0
1,mainline_delay_raw,25543.704003,25543.704003,25543.704003,0.0
2,mainline_delay_clipped,25744.199317,25744.199317,25744.199317,0.0
3,upstream_boundary_delay,0.000000,0.000000,0.000000,0.0
4,local_delay,44806.729702,44806.729702,44806.729702,0.0
5,fairness_penalty,143.623653,143.623653,143.623653,0.0
6,doorway_penalty,0.000000,0.000000,0.000000,0.0
7,safe_penalty,0.000000,0.000000,0.000000,0.0
8,physical_penalty,0.000000,0.000000,0.000000,0.0
9,spillback_penalty,59728.256696,59728.256696,59728.256696,0.0


Max abs reproduction diff vs benchmark run: 0.0
fixed-policy raw objective (scale 1.0) = 130422.809368
published benchmark  raw objective     = 130422.809368
PASS: fixed policy reproduces the benchmark exactly (receiving-aware CTM, LINEAR spillback).



## Historical-demand ablation

This ablation uses:

```text
arrival_multiplier = 1.0
release_scale <= 1.0
```

So ramp arrivals equal the observed ramp release profile. This tests the narrow question:

> Can a simple uniform ramp-choking rule explain mainline-delay improvements by itself?

Because this is a choking ablation, release scales above `1.0` are not included here.


In [16]:

historical_arrival_multiplier = 1.0
historical_release_scales = [round(float(x), 4) for x in np.arange(1.0, 0.9499, -0.0025)]
historical_release_scales += [0.90, 0.85, 0.80]
historical_release_scales = sorted(set(historical_release_scales), reverse=True)

historical_fixed_policy_df, historical_fixed_policy_histories = run_policy_grid(
    arrival_multiplier=historical_arrival_multiplier,
    release_scales=historical_release_scales,
    experiment_name="historical_demand_choking",
)

historical_display_cols = [
    "release_scale",
    "mainline_delay",
    "mainline_delay_change_pct",
    "local_delay",
    "fairness_penalty",
    "doorway_penalty",
    "spillback_penalty",
    "final_external_spillback_queue_B",
    "raw_objective",
    "served_fraction",
]

print("Historical-demand fixed-policy ablation")
display(historical_fixed_policy_df[historical_display_cols].round(6))

historical_no_spillback_df = historical_fixed_policy_df[
    historical_fixed_policy_df["final_external_spillback_queue_B"].abs() < 1e-8
].copy()

best_historical_no_spillback = historical_no_spillback_df.sort_values("mainline_delay").iloc[0]
print("Best historical-demand choking policy with zero final external spillback:")
display(best_historical_no_spillback[historical_display_cols].to_frame("value").round(6))


Historical-demand fixed-policy ablation


,release_scale,mainline_delay,mainline_delay_change_pct,local_delay,fairness_penalty,doorway_penalty,spillback_penalty,final_external_spillback_queue_B,raw_objective,served_fraction
0,1.0000,25744.199317,0.000000,149.729702,4.298249,0.0,0.000000,0.000000,25898.227267,0.998140
1,0.9975,25425.163626,-1.239253,701.140700,9.510715,0.0,0.000000,0.000000,26135.815041,0.995724
2,0.9950,25106.127935,-2.478505,1252.551698,18.250621,0.0,0.000000,0.000000,26376.930255,0.993308
3,0.9925,24787.092245,-3.717758,1803.962697,30.517968,0.0,0.000000,0.000000,26621.572909,0.990892
4,0.9900,24468.056554,-4.957011,2355.373695,46.312756,0.0,0.000000,0.000000,26869.743005,0.988476
5,0.9875,24149.020917,-6.196263,2906.784693,65.634984,0.0,0.000000,0.000000,27121.440595,0.986060
6,0.9850,23829.985363,-7.435516,3458.195691,88.484653,0.0,0.000000,0.000000,27376.665708,0.983644
7,0.9825,23510.949810,-8.674768,4009.606690,114.861763,0.0,0.000000,0.000000,27635.418262,0.981228
8,0.9800,23191.914277,-9.914020,4561.017688,144.766313,0.0,0.000000,0.000000,27897.698278,0.978812
9,0.9775,22872.846074,-11.153399,5112.518886,178.221685,0.0,0.000000,0.000000,28163.586645,0.976395


Best historical-demand choking policy with zero final external spillback:


,value
release_scale,0.9775
mainline_delay,22872.846074
mainline_delay_change_pct,-11.153399
local_delay,5112.518886
fairness_penalty,178.221685
doorway_penalty,0.0
spillback_penalty,0.0
final_external_spillback_queue_B,0.0
raw_objective,28163.586645
served_fraction,0.976395



## Official stress-demand ablation


In [17]:

stress_arrival_multiplier = shared_arrival_multiplier
stress_release_scales = [2.00, 1.50, 1.30, 1.20, 1.15, 1.10, 1.05, 1.00, 0.98, 0.95, 0.90, 0.85, 0.80]

stress_fixed_policy_df, stress_fixed_policy_histories = run_policy_grid(
    arrival_multiplier=stress_arrival_multiplier,
    release_scales=stress_release_scales,
    experiment_name="official_stress_demand",
)

stress_display_cols = [
    "release_scale",
    "mainline_delay",
    "mainline_delay_change_pct",
    "local_delay",
    "fairness_penalty",
    "doorway_penalty",
    "spillback_penalty",
    "final_external_spillback_queue_B",
    "raw_objective",
    "raw_objective_change_pct",
    "served_fraction",
]

print("Official stress-demand fixed-policy ablation")
display(stress_fixed_policy_df[stress_display_cols].round(6))

stress_best_objective = stress_fixed_policy_df.sort_values("raw_objective").iloc[0]
stress_best_mainline = stress_fixed_policy_df.sort_values("mainline_delay").iloc[0]

print("Best stress-demand fixed policy by raw objective:")
display(stress_best_objective[stress_display_cols].to_frame("value").round(6))

print("Best stress-demand fixed policy by mainline delay:")
display(stress_best_mainline[stress_display_cols].to_frame("value").round(6))


Official stress-demand fixed-policy ablation


,release_scale,mainline_delay,mainline_delay_change_pct,local_delay,fairness_penalty,doorway_penalty,spillback_penalty,final_external_spillback_queue_B,raw_objective,raw_objective_change_pct,served_fraction
0,2.00,52426.338287,103.643305,314.726302,26.080527,0.0,0.000000,0.000000,199139.229594,52.687425,1.000000
1,1.50,52423.424566,103.631987,320.696137,26.368352,0.0,0.000000,0.000000,199142.573510,52.689989,1.000000
2,1.30,52318.070955,103.222754,542.367584,45.024274,0.0,0.000000,0.000000,199277.536779,52.793471,0.999966
3,1.20,51398.912232,99.652402,2606.597293,442.369580,0.0,92.155328,2.242387,196393.314184,50.582030,0.985991
4,1.15,44709.077970,73.666609,12789.359761,323.064112,0.0,6283.197464,104.893119,155269.945214,19.051220,0.948761
5,1.10,38399.822181,49.159124,23011.623185,214.095980,0.0,20044.591946,242.400935,83436.417067,-36.026208,0.911411
6,1.05,32122.758406,24.776685,33784.432335,171.068114,0.0,38980.797840,415.324066,105059.056696,-19.447329,0.871996
7,1.00,25744.199317,0.000000,44806.729702,143.623653,0.0,59728.256696,591.742619,130422.809368,0.000000,0.831783
8,0.98,23191.914277,-9.914020,49218.017688,134.667822,0.0,68193.193293,662.406937,140737.793080,7.908880,0.815676
9,0.95,19361.197205,-24.793943,55841.261176,122.292049,0.0,81000.934892,768.664878,156325.685322,19.860695,0.791456


Best stress-demand fixed policy by raw objective:


,value
release_scale,1.1
mainline_delay,38399.822181
mainline_delay_change_pct,49.159124
local_delay,23011.623185
fairness_penalty,214.09598
doorway_penalty,0.0
spillback_penalty,20044.591946
final_external_spillback_queue_B,242.400935
raw_objective,83436.417067
raw_objective_change_pct,-36.026208


Best stress-demand fixed policy by mainline delay:


,value
release_scale,0.8
mainline_delay,3992.933116
mainline_delay_change_pct,-84.48997
local_delay,89314.0
fairness_penalty,82.231552
doorway_penalty,0.0
spillback_penalty,146634.081252
final_external_spillback_queue_B,1316.142777
raw_objective,240023.24592
raw_objective_change_pct,84.034715


In [18]:
# Results are kept as in-notebook tables only. No CSV files, no result pickle,

print(" Fixed-policy reproduction check (vs fixed benchmark) ")
display(reproduction_check_df.round(6))

print("\n Historical-demand ablation (arrival_multiplier = 1.0) ")
display(historical_fixed_policy_df.round(6))

print("\n Official stress-demand ablation (arrival_multiplier = 1.2) ")
display(stress_fixed_policy_df.round(6))

selected_policy_rows = [
    {"label": "historical_best_no_final_spillback", **best_historical_no_spillback.to_dict()},
    {"label": "stress_best_raw_objective", **stress_best_objective.to_dict()},
    {"label": "stress_best_mainline_delay", **stress_best_mainline.to_dict()},
]
selected_policy_df = pd.DataFrame(selected_policy_rows)
print("\n Selected baseline policies ")
display(selected_policy_df.round(6))

# Results remain available in-memory in this namespace; nothing is written to disk.
fixed_policy_results = {
    "benchmark_version": shared.get("version"),
    "benchmark_arrival_multiplier": shared_arrival_multiplier,
    "reproduction_check_df": reproduction_check_df,
    "historical_fixed_policy_df": historical_fixed_policy_df,
    "stress_fixed_policy_df": stress_fixed_policy_df,
    "selected_policy_df": selected_policy_df,
}

print("\nDone. Results are in-notebook only; no files were written.")


 Fixed-policy reproduction check (vs fixed benchmark) 


,metric,fixed_policy_scale1,benchmark_run,published_official,diff_vs_benchmark_run
0,mainline_delay,25744.199317,25744.199317,25744.199317,0.0
1,mainline_delay_raw,25543.704003,25543.704003,25543.704003,0.0
2,mainline_delay_clipped,25744.199317,25744.199317,25744.199317,0.0
3,upstream_boundary_delay,0.000000,0.000000,0.000000,0.0
4,local_delay,44806.729702,44806.729702,44806.729702,0.0
5,fairness_penalty,143.623653,143.623653,143.623653,0.0
6,doorway_penalty,0.000000,0.000000,0.000000,0.0
7,safe_penalty,0.000000,0.000000,0.000000,0.0
8,physical_penalty,0.000000,0.000000,0.000000,0.0
9,spillback_penalty,59728.256696,59728.256696,59728.256696,0.0



 Historical-demand ablation (arrival_multiplier = 1.0) 


,experiment,arrival_multiplier,release_scale,mainline_delay,mainline_delay_raw,mainline_delay_clipped,upstream_boundary_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,raw_objective,sum_history_total_objective,initial_physical_ramp_queue_R,initial_external_spillback_queue_B,total_ramp_arrivals,total_actual_release,final_physical_ramp_queue_R,final_external_spillback_queue_B,ramp_mass_residual,total_ramp_demand_to_account,served_fraction,final_upstream_boundary_queue,mainline_delay_change,mainline_delay_change_pct,local_delay_change,local_delay_change_pct,fairness_penalty_change,fairness_penalty_change_pct,doorway_penalty_change,doorway_penalty_change_pct,spillback_penalty_change,spillback_penalty_change_pct,capacity_penalty_change,capacity_penalty_change_pct,raw_objective_change,raw_objective_change_pct,served_fraction_change,served_fraction_change_pct
0,historical_demand_choking,1.0,1.0000,25744.199317,25543.704003,25744.199317,0.0,149.729702,4.298249,0.0,0.0,0.0,0.000000,0.000000,25898.227267,25898.227267,0.0,0.0,3656.0,3649.200159,6.799841,0.000000,-0.0,3656.0,0.998140,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,0.000000,NaN,0.000000,NaN,0.000000,0.000000,0.000000,0.000000
1,historical_demand_choking,1.0,0.9975,25425.163626,25224.660928,25425.163626,0.0,701.140700,9.510715,0.0,0.0,0.0,0.000000,0.000000,26135.815041,26135.815041,0.0,0.0,3656.0,3640.367119,15.632881,0.000000,-0.0,3656.0,0.995724,0.0,-319.035691,-1.239253,551.410998,368.270952,5.212466,121.269526,0.0,NaN,0.000000,NaN,0.000000,NaN,237.587773,0.917390,-0.002416,-0.242054
2,historical_demand_choking,1.0,0.9950,25106.127935,24905.617854,25106.127935,0.0,1252.551698,18.250621,0.0,0.0,0.0,0.000000,0.000000,26376.930255,26376.930255,0.0,0.0,3656.0,3631.534079,24.465921,0.000000,0.0,3656.0,0.993308,0.0,-638.071381,-2.478505,1102.821996,736.541903,13.952372,324.605981,0.0,NaN,0.000000,NaN,0.000000,NaN,478.702987,1.848401,-0.004832,-0.484108
3,historical_demand_choking,1.0,0.9925,24787.092245,24586.574779,24787.092245,0.0,1803.962697,30.517968,0.0,0.0,0.0,0.000000,0.000000,26621.572909,26621.572909,0.0,0.0,3656.0,3622.701039,33.298961,0.000000,-0.0,3656.0,0.990892,0.0,-957.107072,-3.717758,1654.232995,1104.812855,26.219719,610.009363,0.0,NaN,0.000000,NaN,0.000000,NaN,723.345642,2.793031,-0.007248,-0.726162
4,historical_demand_choking,1.0,0.9900,24468.056554,24267.531704,24468.056554,0.0,2355.373695,46.312756,0.0,0.0,0.0,0.000000,0.000000,26869.743005,26869.743005,0.0,0.0,3656.0,3613.868000,42.132000,0.000000,-0.0,3656.0,0.988476,0.0,-1276.142763,-4.957011,2205.643993,1473.083806,42.014507,977.479672,0.0,NaN,0.000000,NaN,0.000000,NaN,971.515737,3.751283,-0.009664,-0.968217
5,historical_demand_choking,1.0,0.9875,24149.020917,23948.488629,24149.020917,0.0,2906.784693,65.634984,0.0,0.0,0.0,0.000000,0.000000,27121.440595,27121.440595,0.0,0.0,3656.0,3605.034960,50.965040,0.000000,0.0,3656.0,0.986060,0.0,-1595.178399,-6.196263,2757.054991,1841.354758,61.336736,1427.016910,0.0,NaN,0.000000,NaN,0.000000,NaN,1223.213327,4.723155,-0.012080,-1.210271
6,historical_demand_choking,1.0,0.9850,23829.985363,23629.445554,23829.985363,0.0,3458.195691,88.484653,0.0,0.0,0.0,0.000000,0.000000,27376.665708,27376.665708,0.0,0.0,3656.0,3596.201920,59.798080,0.000000,-0.0,3656.0,0.983644,0.0,-1914.213953,-7.435516,3308.465989,2209.625710,84.186405,1958.621075,0.0,NaN,0.000000,NaN,0.000000,NaN,1478.438441,5.708647,-0.014496,-1.452325
7,historical_demand_choking,1.0,0.9825,23510.949810,23310.402479,23510.949810,0.0,4009.606690,114.861763,0.0,0.0,0.0,0.000000,0.000000,27635.418262,27635.418262,0.0,0.0,3656.0,3587.368881,68.631119,0.000000,-0.0,3656.0,0.981228,0.0,-2233.249507,-8.674768,3859.876988,2577.896661,110.563514,2572.292168,0.0,NaN,0.000000,NaN,0.000000,NaN,1737.190995,6.707760,-0.016912,-1.694379
8,historical_demand_choking,1.0,0.9800,23191.914277,22991.359404,23191.914277,0.0,4561.017688,144.766313,0.


 Official stress-demand ablation (arrival_multiplier = 1.2) 


,experiment,arrival_multiplier,release_scale,mainline_delay,mainline_delay_raw,mainline_delay_clipped,upstream_boundary_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,raw_objective,sum_history_total_objective,initial_physical_ramp_queue_R,initial_external_spillback_queue_B,total_ramp_arrivals,total_actual_release,final_physical_ramp_queue_R,final_external_spillback_queue_B,ramp_mass_residual,total_ramp_demand_to_account,served_fraction,final_upstream_boundary_queue,mainline_delay_change,mainline_delay_change_pct,local_delay_change,local_delay_change_pct,fairness_penalty_change,fairness_penalty_change_pct,doorway_penalty_change,doorway_penalty_change_pct,spillback_penalty_change,spillback_penalty_change_pct,capacity_penalty_change,capacity_penalty_change_pct,raw_objective_change,raw_objective_change_pct,served_fraction_change,served_fraction_change_pct
0,official_stress_demand,1.2,2.00,52426.338287,52230.385870,52426.338287,0.0,314.726302,26.080527,0.0,146372.084478,0.0,0.000000,146372.084478,199139.229594,199139.229594,0.0,0.0,4387.2,4387.200000,0.000000,0.000000,0.0,4387.2,1.000000,0.0,26682.138970,103.643305,-44492.003400,-99.297591,-117.543126,-81.841064,0.0,NaN,-59728.256696,-100.000000,86643.827782,145.063380,68716.420227,52.687425,0.168217,20.223605
1,official_stress_demand,1.2,1.50,52423.424566,52227.500028,52423.424566,0.0,320.696137,26.368352,0.0,146372.084456,0.0,0.000000,146372.084456,199142.573510,199142.573510,0.0,0.0,4387.2,4387.200000,0.000000,0.000000,0.0,4387.2,1.000000,0.0,26679.225250,103.631987,-44486.033565,-99.284268,-117.255301,-81.640662,0.0,NaN,-59728.256696,-100.000000,86643.827760,145.063380,68719.764142,52.689989,0.168217,20.223605
2,official_stress_demand,1.2,1.30,52318.070955,52122.382327,52318.070955,0.0,542.367584,45.024274,0.0,146372.073965,0.0,0.000000,146372.073965,199277.536779,199277.536779,0.0,0.0,4387.2,4387.051757,0.148243,0.000000,0.0,4387.2,0.999966,0.0,26573.871639,103.222754,-44264.362118,-98.789540,-98.599379,-68.651212,0.0,NaN,-59728.256696,-100.000000,86643.817269,145.063362,68854.727411,52.793471,0.168183,20.219543
3,official_stress_demand,1.2,1.20,51398.912232,51202.814849,51398.912232,0.0,2606.597293,442.369580,0.0,141853.279752,0.0,92.155328,141945.435079,196393.314184,196393.314184,0.0,0.0,4387.2,4325.739833,59.217780,2.242387,0.0,4387.2,0.985991,0.0,25654.712915,99.652402,-42200.132409,-94.182576,298.745927,208.006078,0.0,NaN,-59636.101369,-99.845709,82217.178383,137.652064,65970.504817,50.582030,0.154208,18.539396
4,official_stress_demand,1.2,1.15,44709.077970,44511.919482,44709.077970,0.0,12789.359761,323.064112,0.0,91165.245906,0.0,6283.197464,97448.443371,155269.945214,155269.945214,0.0,0.0,4387.2,4162.403335,119.903546,104.893119,-0.0,4387.2,0.948761,0.0,18964.878653,73.666609,-32017.369941,-71.456610,179.440459,124.937958,0.0,NaN,-53445.059232,-89.480360,37720.186674,63.153001,24847.135846,19.051220,0.116977,14.063443
5,official_stress_demand,1.2,1.10,38399.822181,38199.788839,38399.822181,0.0,23011.623185,214.095980,0.0,1766.283775,0.0,20044.591946,21810.875721,83436.417067,83436.417067,0.0,0.0,4387.2,3998.541843,146.257223,242.400935,-0.0,4387.2,0.911411,0.0,12655.622864,49.159124,-21795.106517,-48.642484,70.472327,49.067355,0.0,NaN,-39683.664751,-66.440353,-37917.380975,-63.483154,-46986.392300,-36.026208,0.079627,9.573103
6,official_stress_demand,1.2,1.05,32122.758406,31922.408844,32122.758406,0.0,33784.432335,171.068114,0.0,0.000000,0.0,38980.797840,38980.797840,105059.056696,105059.056696,0.0,0.0,4387.2,3825.618711,146.257223,415.324066,-0.0,4387.2,0.871996,0.0,6378.559090,24.776685,-11022.297367,-24.599647,27.444461,19.108594,0.0,NaN,-20747.458856,-34.736421,-20747.458856,-34.736421,-25363.752672,-19.447329,0.040212,4.834444
7,official_stress_demand,1.2,1.00,25744.199317,25543.704003,25744.199317,0.0,44806.729702,143.623653,0.0,0.000000,0.0,59728.256696,59728.256696,130422.809368,130422.809368,


 Selected baseline policies 


,label,experiment,arrival_multiplier,release_scale,mainline_delay,mainline_delay_raw,mainline_delay_clipped,upstream_boundary_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,raw_objective,sum_history_total_objective,initial_physical_ramp_queue_R,initial_external_spillback_queue_B,total_ramp_arrivals,total_actual_release,final_physical_ramp_queue_R,final_external_spillback_queue_B,ramp_mass_residual,total_ramp_demand_to_account,served_fraction,final_upstream_boundary_queue,mainline_delay_change,mainline_delay_change_pct,local_delay_change,local_delay_change_pct,fairness_penalty_change,fairness_penalty_change_pct,doorway_penalty_change,doorway_penalty_change_pct,spillback_penalty_change,spillback_penalty_change_pct,capacity_penalty_change,capacity_penalty_change_pct,raw_objective_change,raw_objective_change_pct,served_fraction_change,served_fraction_change_pct
0,historical_best_no_final_spillback,historical_demand_choking,1.0,0.9775,22872.846074,22672.283543,22872.846074,0.0,5112.518886,178.221685,0.0,0.000000,0.0,0.000000,0.000000,28163.586645,28163.586645,0.0,0.0,3656.0,3569.699074,86.300926,0.000000,-0.0,3656.0,0.976395,0.0,-2871.353243,-11.153399,4962.789185,3314.498807,173.923436,4046.379096,0.0,NaN,0.000000,NaN,0.000000,NaN,2265.359378,8.747160,-0.021745,-2.178589
1,stress_best_raw_objective,official_stress_demand,1.2,1.1000,38399.822181,38199.788839,38399.822181,0.0,23011.623185,214.095980,0.0,1766.283775,0.0,20044.591946,21810.875721,83436.417067,83436.417067,0.0,0.0,4387.2,3998.541843,146.257223,242.400935,-0.0,4387.2,0.911411,0.0,12655.622864,49.159124,-21795.106517,-48.642484,70.472327,49.067355,0.0,NaN,-39683.664751,-66.440353,-37917.380975,-63.483154,-46986.392300,-36.026208,0.079627,9.573103
2,stress_best_mainline_delay,official_stress_demand,1.2,0.8000,3992.933116,3789.036203,3992.933116,0.0,89314.000000,82.231552,0.0,0.000000,0.0,146634.081252,146634.081252,240023.245920,240023.245920,0.0,0.0,4387.2,2924.800000,146.257223,1316.142777,-0.0,4387.2,0.666667,0.0,-21751.266200,-84.489970,44507.270298,99.331664,-61.392101,-42.745119,0.0,NaN,86905.824556,145.502028,86905.824556,145.502028,109600.436552,84.034715,-0.165117,-19.850930



Done. Results are in-notebook only; no files were written.
